# Lịch chạy và biểu đồ
Notebook này dành cho máy chạy thí nghiệm. Chỉ bắt đầu sau khi pipeline đã chạy thành công trên dữ liệu thật.
Kế hoạch baseline có 45 lượt, CAFIN/ablation có 30 lượt. Tạo plan không chạy huấn luyện.
Mỗi lệnh sweep-run mặc định chạy thêm một lượt; các lượt đã hoàn thành được kiểm tra và bỏ qua.
CAFIN là thiết kế mới v1, chưa phải mô hình được chứng minh tốt hơn hoặc mới trong literature.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

"""Standard-library helpers embedded in Colab cells before source is extracted."""

import hashlib
from pathlib import Path
import tempfile
import zipfile
import json


def file_sha256(path):
    checksum = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            checksum.update(chunk)
    return checksum.hexdigest()


def restore_project(archive_path, destination):
    """Publish a complete extraction, or reject stale/modified local source."""
    archive_path, destination = Path(archive_path), Path(destination)
    archive_hash = file_sha256(archive_path)
    receipt_name = '.cafin_archive.json'
    receipt = destination / receipt_name
    restart = 'Start a fresh runtime and rerun setup with the intended code ZIP.'
    if destination.exists():
        if not receipt.is_file():
            raise ValueError('Existing project has no completed extraction receipt. ' + restart)
        recorded = json.loads(receipt.read_text(encoding='utf-8'))
        if recorded['archive_sha256'] != archive_hash:
            raise ValueError('Code ZIP changed since extraction. ' + restart)
        for name, expected in recorded['files'].items():
            target = (destination / name).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError('Unsafe receipt path')
            if not target.is_file() or file_sha256(target) != expected:
                raise ValueError('Extracted project file changed: ' + name + '. ' + restart)
        return recorded

    destination.parent.mkdir(parents=True, exist_ok=True)
    # Stage on the same filesystem; a failed extraction never publishes PROJECT.
    with tempfile.TemporaryDirectory(prefix='cafin_extract_', dir=destination.parent) as temporary:
        stage = Path(temporary) / 'project'
        stage.mkdir()
        with zipfile.ZipFile(archive_path) as archive:
            names = archive.namelist()
            if len(names) != len(set(names)) or receipt_name in names:
                raise ValueError('Duplicate or reserved archive member')
            for member in archive.infolist():
                if not (stage / member.filename).resolve().is_relative_to(stage.resolve()):
                    raise ValueError('Unsafe archive path')
            for required in ('pyproject.toml', 'src/cafin/__init__.py', 'src/cafin/cli.py'):
                if required not in names:
                    raise ValueError('Incomplete code ZIP: missing ' + required)
            archive.extractall(stage)
            files = {member.filename: file_sha256(stage / member.filename)
                     for member in archive.infolist() if not member.is_dir()}
        recorded = {'archive_sha256': archive_hash, 'files': files}
        (stage / receipt_name).write_text(json.dumps(recorded, indent=2), encoding='utf-8')
        stage.rename(destination)
    return recorded


DRIVE_ROOT = Path('/content/drive/MyDrive/CAFIN')
PROJECT_ZIP = DRIVE_ROOT / 'cafin_ipinyou_code.zip'
PROJECT = Path('/content/cafin_project')
source_receipt = restore_project(PROJECT_ZIP, PROJECT)
print('Code ZIP SHA-256:', source_receipt['archive_sha256'])
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('nvidia-smi unavailable; CPU checks can still run.')

def run_cli(*args):
    subprocess.run([sys.executable, '-m', 'cafin', *map(str, args)], check=True)

def save_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2), encoding='utf-8')

In [ ]:
import shutil
DATA_DIR = Path('/content/cafin_data/standardized')
CACHE = DRIVE_ROOT / 'cache' / 'standardized.zip'
if not (DATA_DIR / 'manifest.json').exists() and CACHE.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CACHE) as archive:
        for member in archive.infolist():
            if not (DATA_DIR / member.filename).resolve().is_relative_to(DATA_DIR.resolve()):
                raise ValueError('Unsafe cache path')
        archive.extractall(DATA_DIR)
print('Prepared data:', (DATA_DIR / 'manifest.json').exists())

In [ ]:
EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments' / 'cafin_v1'
PLAN = EXPERIMENT_ROOT / 'plan.json'
SPEC = PROJECT / 'configs/sweep_cafin.json'
BASE = PROJECT / 'configs/train_cafin.json'
run_cli('sweep-plan', '--spec', SPEC, '--base', BASE, '--output', PLAN)
plan = json.loads(PLAN.read_text())
print('Declared runs:', len(plan['entries']))

In [ ]:
RUN_SWEEP = False
if RUN_SWEEP:
    run_cli('sweep-run', '--plan', PLAN, '--data-dir', DATA_DIR,
            '--run-root', EXPERIMENT_ROOT/'runs', '--max-runs', 1)
else:
    print('Plan created only. Set RUN_SWEEP=True to run/resume one validation experiment.')

## Diagnostics và xuất hình
Chỉ chạy sau khi đã có kết quả. Các cell dưới dùng validation; chúng không mở test.
Để tạo hình final, dùng các artifacts test đã được tạo qua quy trình freeze/evaluate.
JSD đo phân bố đặc trưng đã encode; UNK gộp rare/missing/unseen. Error bars là sample std, không phải confidence intervals.

In [ ]:
REPORTS = EXPERIMENT_ROOT / 'results'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[plots]'], check=True)
if list((EXPERIMENT_ROOT/'runs').glob('*/valid_metrics.json')):
    run_cli('summarize', '--runs', EXPERIMENT_ROOT/'runs', '--split', 'valid', '--output', REPORTS/'validation_summary.csv')
    run_cli('plot', 'ctr', '--input', REPORTS/'validation_summary.csv', '--output', REPORTS/'figures')
else:
    print('No validation runs yet; no CTR figures generated.')
if (DATA_DIR/'manifest.json').exists():
    run_cli('drift', '--data-dir', DATA_DIR, '--split', 'valid', '--output', REPORTS/'drift_valid.json')
    run_cli('plot', 'drift', '--input', REPORTS/'drift_valid.json', '--output', REPORTS/'figures')

Nếu dùng sweep, run IDs có hash cấu hình, không giống tên run thủ công trong notebook 01. Khi dùng notebook 02 cho final test, đặt RUN_DIR đúng thư mục cụ thể trong experiments/.../runs/.
Raw RTB cần chuẩn bị chronological data và bidder policy như README. Hình budget–click lấy từ các rtb_results.json đã xuất bằng lệnh replay; không tạo số liệu thay thế.